In [1]:
import os
os.environ["MUJOCO_GL"] = "egl"

import sys
from datetime import datetime
from pathlib import Path

import imageio
import numpy as np
import stable_worldmodel as swm
import torch
from einops import rearrange
from omegaconf import OmegaConf
from stable_pretraining.data.dataset_stats import ImageNet

from IPython.display import display
from PIL.Image import fromarray
from eval_plain import encode, to_model_pixels, sample_start_goal_rows, plan, to_model_pixels

/home/yvasiliuk/anaconda3/envs/fast-lewm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CONFIG_DIR = Path(os.path.abspath('./config/eval'))

def load_cfg(env: str = "pusht"):
    cfg = OmegaConf.load(CONFIG_DIR / f"{env}.yaml")
    cfg.pop("defaults")
    cfg.solver = OmegaConf.load(CONFIG_DIR / "solver" / "cem.yaml")

    # --- knobs ---------------------------------------------------------------
    cfg.cache_dir = os.path.expanduser("~/stablewm_data")
    cfg.eval.num_eval = 50  # episodes = parallel envs
    cfg.eval.goal_offset_steps = 25  # goal = expert frame this many steps after start
    cfg.eval.eval_budget = 50  # env steps allowed to reach the goal
    cfg.solver.num_samples = 300  # CEM candidates per iteration
    cfg.solver.n_steps = 30  # CEM iterations per replan
    cfg.solver.topk = 30  # CEM elites
    cfg.save_video = True
    return cfg

print(CONFIG_DIR)

/home/yvasiliuk/Fast-LeWorldModel/config/eval


In [3]:
env = 'pusht'
cfg = load_cfg(env)
N, budget = cfg.eval.num_eval, cfg.eval.eval_budget
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
run_dir = Path('results') / f"eval-{env}-{datetime.now():%Y%m%d-%H%M%S}"
run_dir.mkdir(parents=True, exist_ok=True)

In [4]:
ckpt_path = os.path.abspath('./weights/Fast-lewm_pusht_object.ckpt')
model = torch.load(ckpt_path, weights_only=False, map_location=device)
model.eval().requires_grad_(False)

dataset = swm.data.HDF5Dataset(cfg.eval.dataset_name, cache_dir=cfg.cache_dir)
start_rows, goal_rows = sample_start_goal_rows(dataset, cfg)
start = dataset.get_row_data(start_rows)
goal = dataset.get_row_data(goal_rows)
actions0 = dataset.get_col_data("action").astype(np.float64)
actions = np.copy(actions0)
actions = actions[~np.isnan(actions).any(1)]
act_mean, act_std = actions.mean(0), actions.std(0)  # = sklearn StandardScaler
cfg.act_dim = actions.shape[1]

print(act_mean, act_std)

[-0.00781256  0.00686069] [0.20846744 0.20674863]


In [5]:
# with imageio.get_writer("start.mp4", fps=2, codec="libx264") as w:
#     for iidx, (frame_start, frame_goal) in enumerate(zip(start['pixels'], goal['pixels'])):
#         w.append_data(np.hstack([frame_start, frame_goal]))

In [6]:
cfg.world.max_episode_steps = 2 * budget
print(f'cfg.world.max_episode_steps: {cfg.world.max_episode_steps}')
world = swm.World(**OmegaConf.to_container(cfg.world, resolve=True), image_shape=(224, 224))
world.reset(seed=None, options=[{} for _ in range(N)])
values = {**start, **{f"goal_{k}": v for k, v in goal.items()}}

cfg.world.max_episode_steps: 100
16:25:20.760 | INFO    (95676, stable_worldmodel.world) | 🌍🌍🌍 World swm/PushT-v1 initialized 🌍🌍🌍
16:25:20.760 | INFO    (95676, stable_worldmodel.world) | 🕹️ 🕹️ 🕹️ Action space 🕹️ 🕹️ 🕹️
16:25:20.761 | INFO    (95676, stable_worldmodel.world) | Box(-1.0, 1.0, (50, 2), float32)
16:25:20.761 | INFO    (95676, stable_worldmodel.world) | 👁️ 👁️ 👁️ Observation space 👁️ 👁️ 👁️
16:25:20.762 | INFO    (95676, stable_worldmodel.world) | Dict('proprio': Box(0.0, 512.0, (50, 4), float64), 'state': Box(0.0, [[512.         512.         512.         512.           6.28318531
  512.         512.        ]
 [512.         512.         512.         512.           6.28318531
  512.         512.        ]
 [512.         512.         512.         512.           6.28318531
  512.         512.        ]
 [512.         512.         512.         512.           6.28318531
  512.         512.        ]
 [512.         512.         512.         512.           6.28318531
  512.         512

agent:
    angle: Box(-6.283185307179586, 6.283185307179586, (), float64)
    color: Box(0, 255, (3,), uint8)
    scale: Box(20.0, 60.0, (), float32)
    shape: Discrete(8)
    start_position: Box(50.0, 450.0, (2,), float64)
    velocity: Box(0.0, 512.0, (2,), float64)
background:
    color: Box(0, 255, (3,), uint8)
block:
    angle: Box(-6.283185307179586, 6.283185307179586, (), float64)
    color: Box(0, 255, (3,), uint8)
    scale: Box(20.0, 60.0, (), float32)
    shape: Discrete(7, start=1)
    start_position: Box(100.0, 400.0, (2,), float64)
goal:
    angle: Box(-6.283185307179586, 6.283185307179586, (), float64)
    color: Box(0, 255, (3,), uint8)
    position: Box(50.0, 450.0, (2,), float64)
    scale: Box(20.0, 60.0, (), float32)
rendering:
    render_goal: Discrete(2)

/home/yvasiliuk/anaconda3/envs/fast-lewm/lib/python3.10/site-packages/gymnasium/spaces/box.py:424: UserWarning: WARN: Casting input x to numpy array.
  gym.logger.warn("Casting input x to numpy array.")


In [ ]:
for i, e in enumerate(world.envs.unwrapped.envs):
    for spec in cfg.eval.callables:  # e.g. _set_state(state=...), _set_goal_state(goal_state=...)
        kwargs = {
            name: values[arg.value][i].copy() if arg.get("in_dataset", True) else arg.value
            for name, arg in spec.args.items()
        }
        getattr(e.unwrapped, spec.method)(**kwargs)

In [ ]:
gen = torch.Generator(device=device).manual_seed(int(cfg.solver.seed))
z_goal = encode(model, to_model_pixels(goal["pixels"], device))  # goal is fixed
obs = start["pixels"]  # first observation = dataset frame, as in evaluate_from_dataset
R = cfg.plan_config.receding_horizon
buffer, warm, plan_costs = [], None, []
success = np.zeros(N, dtype=bool)
frames = np.empty((N, budget + 1, *obs.shape[1:]), dtype=np.uint8)

In [ ]:
frames.shape

In [ ]:
obs.shape

In [ ]:
# display(fromarray(obs[0]))
# display(fromarray(obs[24]))
# display(fromarray(obs[-1]))

In [ ]:
buffer

In [ ]:
t = 0
frames[:, t] = obs
print(not buffer)

In [ ]:
# encode start pixels here
z0 = encode(model, to_model_pixels(obs, device))

In [ ]:
N, device = z0.size(0), z0.device
S, K = cfg.solver.num_samples, cfg.solver.topk
H = cfg.plan_config.horizon

print(N, S, K, H)

In [ ]:
blocks = cfg.plan_config.action_num_blocks
plan_dim = cfg.act_dim * blocks * cfg.plan_config.action_block_size
plan_dim

In [ ]:
mean = torch.zeros(N, H, plan_dim, device=device)
mean.shape

In [ ]:
std = torch.full_like(mean, float(cfg.solver.var_scale))
cfg.solver.var_scale

In [ ]:
z_goal.shape

In [ ]:
z0_s = z0.repeat_interleave(S, 0)  # one copy per candidate: (N*S, D)
zg_s = z_goal.repeat_interleave(S, 0)
zg_s.shape

In [ ]:
cfg.solver.n_steps

In [ ]:
cand = torch.randn(N, S, H, plan_dim, generator=gen, device=device)
cand = cand * std[:, None] + mean[:, None]
cand[:, 0] = mean  # always keep the current mean as a candidate
tokens = rearrange(cand, "n s h (f a) -> (n s) h f a", f=blocks)
tokens.shape

In [ ]:
z0_s.shape

In [ ]:
act = model.action_encoder(tokens[:, 0], return_last_only=True, latent=z0_s[:, None])
act.shape

In [ ]:
z_pred =  model.pred_proj(model.predictor(z0_s, act)[:, 0])
z_pred.shape

In [ ]:
zg_s.shape

In [ ]:
cost = (z_pred - zg_s).pow(2).sum(-1).view(N, S)
cost.shape

In [ ]:
elite_cost, idx = cost.topk(K, dim=1, largest=False)
elite_cost

In [ ]:
elite = cand[torch.arange(N, device=device)[:, None], idx]
elite.shape

In [ ]:
elite.mean(1).shape

In [ ]:
elite.std(1).shape

In [ ]:
elite_cost.mean().item()

In [ ]:
elite_cost.shape

In [ ]:
from eval_plain import predict_final

for _ in range(cfg.solver.n_steps):
    cand = torch.randn(N, S, H, plan_dim, generator=gen, device=device)
    cand = cand * std[:, None] + mean[:, None]
    cand[:, 0] = mean  # always keep the current mean as a candidate

    tokens = rearrange(cand, "n s h (f a) -> (n s) h f a", f=blocks)
    z_pred = predict_final(model, z0_s, tokens)
    cost = (z_pred - zg_s).pow(2).sum(-1).view(N, S)  # squared distance to goal embedding

    elite_cost, idx = cost.topk(K, dim=1, largest=False)
    elite = cand[torch.arange(N, device=device)[:, None], idx]  # (N, K, H, plan_dim)
    mean, std = elite.mean(1), elite.std(1)
    print(elite_cost.mean().item())

In [ ]:
obs.shape

In [ ]:
imgstack = np.concatenate([obs[i] for i in range(0, len(obs), 24)], axis=1)
imgstack.shape

In [ ]:
goal['pixels'].shape

In [ ]:
goal['action'][:5]

In [ ]:
img_stackgoal = np.concatenate( [goal['pixels'][i] for i in range(5)], axis=1 )
display(fromarray(img_stackgoal))

In [ ]:
img_stackgoal = np.concatenate( [goal['pixels'][i] for i in range(5, 5 +len(goal['pixels']) // 10, 1)], axis=1 )
display(fromarray(img_stackgoal))

In [ ]:
display(fromarray(imgstack))

In [ ]:
for t in range(budget):
    frames[:, t] = obs
    if not buffer:
        z0 = encode(model, to_model_pixels(obs, device))
        mean, cost = plan(model, z0, z_goal, cfg, gen, init=warm)
        plan_costs.append(cost.cpu().numpy())
        warm = mean[:, R:]  # warm start: unexecuted tail seeds the next plan
        buffer = list(rearrange(mean[:, :R], "n r (k a) -> (r k) n a", a=cfg.act_dim))

    action = buffer.pop(0).cpu().numpy() * act_std + act_mean  # un-normalize
    _, _, terminated, _, infos = world.envs.step(action.astype(np.float32))
    world.envs.unwrapped._autoreset_envs = np.zeros(N)  # keep envs alive after success
    success |= terminated
    obs = infos["pixels"][:, -1]
    break
frames[:, -1] = obs

In [ ]:
infos.keys()

In [ ]:
action.shape

In [ ]:
infos['reward'].shape

In [ ]:
infos['_pixels']

In [ ]:
infos['_pos_agent']

In [ ]:
metrics = {
    "success_rate": 100.0 * success.mean(),
    "episode_successes": success.astype(int).tolist(),
    "plan_cost_mean": float(np.mean(plan_costs)),
}
print(metrics)

In [ ]:
run_dir

In [ ]:
with open(run_dir / "results.txt", "w") as f:
    f.write(f"ckpt: {ckpt_path}\n{OmegaConf.to_yaml(cfg)}\nmetrics: {metrics}\n")


In [ ]:
cfg.save_video

In [ ]:
for i in range(N):
    with imageio.get_writer(run_dir / f"rollout_{i}.mp4", fps=15, codec="libx264") as w:
        for frame in frames[i]:
            w.append_data(np.hstack([frame, goal["pixels"][i]]))

In [ ]:
pixels_pt = to_model_pixels(obs, device)
pixels_pt.shape

In [ ]:
z0 = encode(model, pixels_pt)

In [ ]:
z0.shape

In [ ]:
mean, cost = plan(model, z0, z_goal, cfg, gen, init=warm)

In [ ]:
plan_costs.append(cost.cpu().numpy())
plan_costs

In [ ]:
warm = mean[:, R:]  # warm start: unexecuted tail seeds the next plan
buffer = list(rearrange(mean[:, :R], "n r (k a) -> (r k) n a", a=cfg.act_dim))

In [ ]:
action = buffer.pop(0).cpu().numpy() * act_std + act_mean

In [ ]:
_, _, terminated, _, infos = world.envs.step(action.astype(np.float32))

In [ ]:
buffer